# Rich display

The outputs of the cells are not limited to text: `display` shows a value with its richest representation,
and helpers show HTML, Markdown, SVG, images, LaTeX or JSON.

## Display helpers

The helpers take a string, or can be used as tagged templates.

In [ ]:
display.html('<h3 style="margin: 0">Hello <em>HTML</em></h3>');
display.markdown('**Markdown** with a [link](https://bellard.org/quickjs/)');
display.latex('$$\\int_0^1 x^2\\,dx = \\frac{1}{3}$$');

In [ ]:
const items = ['QuickJS', 'xeus', 'Jupyter'];
display.markdown`
| # | Project |
|---|---------|
${items.map((item, i) => `| ${i + 1} | ${item} |`).join('\n')}
`;

In [ ]:
display.json({ name: 'xeus-quickjs', features: ['execution', 'completion', 'inspection', 'widgets'], nested: { a: [1, 2, 3] } });

## Rich representation of objects

Objects can define methods returning their representation: `_toHtml`, `_toMarkdown`, `_toSvg`, `_toPng`,
`_toJpeg`, `_toLatex` and `_toJson`, or `_toMime` returning a full MIME bundle. They are used when the
object is the result of a cell, or when it is displayed.

In [ ]:
class Matrix {
  constructor(rows) {
    this.rows = rows;
  }

  multiply(other) {
    return new Matrix(this.rows.map(row =>
      other.rows[0].map((_, j) => row.reduce((sum, value, k) => sum + value * other.rows[k][j], 0))));
  }

  _toLatex() {
    const body = this.rows.map(row => row.join(' & ')).join(' \\\\ ');
    return `$$\\begin{pmatrix} ${body} \\end{pmatrix}$$`;
  }
}

const a = new Matrix([[1, 2], [3, 4]]);
a.multiply(a)

In [ ]:
class Gauge {
  constructor(value, label) {
    this.value = value;
    this.label = label;
  }

  _toSvg() {
    const angle = Math.PI * (1 - this.value / 100);
    const x = 100 + 80 * Math.cos(angle);
    const y = 100 - 80 * Math.sin(angle);
    return `<svg xmlns="http://www.w3.org/2000/svg" width="200" height="120">
      <path d="M 20 100 A 80 80 0 0 1 180 100" fill="none" stroke="#ddd" stroke-width="16"/>
      <path d="M 20 100 A 80 80 0 0 1 ${x.toFixed(1)} ${y.toFixed(1)}" fill="none" stroke="#f7a41d" stroke-width="16"/>
      <text x="100" y="95" text-anchor="middle" font-family="sans-serif" font-size="24">${this.value}%</text>
      <text x="100" y="116" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#666">${this.label}</text>
    </svg>`;
  }
}

display(new Gauge(72, 'CPU'));
display(new Gauge(35, 'Memory'));

The `Jupyter.display` symbol, also used by Deno, works too:

In [ ]:
({
  [Symbol.for('Jupyter.display')]() {
    return { 'text/html': '<b style="color: seagreen">Displayed with the Jupyter.display symbol</b>' };
  }
})

## Charts

SVG makes it easy to draw charts from the data.

In [ ]:
function barChart(data, { width = 420, height = 200, color = '#4e79a7' } = {}) {
  const max = Math.max(...data.map(d => d.value));
  const band = width / data.length;
  const bars = data.map((d, i) => {
    const h = (d.value / max) * (height - 30);
    const x = i * band + 6;
    return `<rect x="${x}" y="${height - 20 - h}" width="${band - 12}" height="${h}" fill="${color}" rx="3"/>
      <text x="${x + (band - 12) / 2}" y="${height - 5}" text-anchor="middle" font-size="11" font-family="sans-serif">${d.label}</text>
      <text x="${x + (band - 12) / 2}" y="${height - 24 - h}" text-anchor="middle" font-size="10" font-family="sans-serif">${d.value}</text>`;
  });
  return display.svg(`<svg xmlns="http://www.w3.org/2000/svg" width="${width}" height="${height}">${bars.join('')}</svg>`);
}

barChart([
  { label: 'Mon', value: 12 }, { label: 'Tue', value: 19 }, { label: 'Wed', value: 7 },
  { label: 'Thu', value: 15 }, { label: 'Fri', value: 22 }, { label: 'Sat', value: 9 }
]);

## Images

`display.png` and `display.jpeg` take the bytes of an image, in a `Uint8Array` or an `ArrayBuffer`, and
`display.image` reads an image file. Here is a small PNG encoder, writing uncompressed images:

In [ ]:
const CRC_TABLE = Array.from({ length: 256 }, (_, n) => {
  let c = n;
  for (let k = 0; k < 8; k++) {
    c = c & 1 ? 0xedb88320 ^ (c >>> 1) : c >>> 1;
  }
  return c >>> 0;
});

function crc32(bytes) {
  let crc = 0xffffffff;
  for (const byte of bytes) {
    crc = CRC_TABLE[(crc ^ byte) & 0xff] ^ (crc >>> 8);
  }
  return (crc ^ 0xffffffff) >>> 0;
}

/** Encodes RGB pixels in a PNG image, without compression. */
function encodePng(width, height, rgb) {
  const stride = width * 3 + 1;
  const raw = new Uint8Array(stride * height);
  for (let y = 0; y < height; y++) {
    raw.set(rgb.subarray(y * width * 3, (y + 1) * width * 3), y * stride + 1);
  }
  // zlib stream made of stored deflate blocks
  const blocks = Math.ceil(raw.length / 65535);
  const zlib = new Uint8Array(2 + raw.length + blocks * 5 + 4);
  zlib.set([0x78, 0x01]);
  let p = 2;
  for (let i = 0; i < raw.length; i += 65535) {
    const length = Math.min(65535, raw.length - i);
    zlib.set([i + length >= raw.length ? 1 : 0, length & 0xff, length >> 8, ~length & 0xff, (~length >> 8) & 0xff], p);
    zlib.set(raw.subarray(i, i + length), p + 5);
    p += 5 + length;
  }
  let a = 1, b = 0;
  for (const byte of raw) {
    a = (a + byte) % 65521;
    b = (b + a) % 65521;
  }
  new DataView(zlib.buffer).setUint32(p, ((b << 16) | a) >>> 0);

  const chunk = (type, data) => {
    const bytes = new Uint8Array(12 + data.length);
    const view = new DataView(bytes.buffer);
    view.setUint32(0, data.length);
    bytes.set([...type].map(c => c.charCodeAt(0)), 4);
    bytes.set(data, 8);
    view.setUint32(8 + data.length, crc32(bytes.subarray(4, 8 + data.length)));
    return bytes;
  };
  const header = new Uint8Array(13);
  new DataView(header.buffer).setUint32(0, width);
  new DataView(header.buffer).setUint32(4, height);
  header.set([8, 2, 0, 0, 0], 8);
  const parts = [new Uint8Array([137, 80, 78, 71, 13, 10, 26, 10]), chunk('IHDR', header), chunk('IDAT', zlib), chunk('IEND', new Uint8Array(0))];
  const png = new Uint8Array(parts.reduce((sum, part) => sum + part.length, 0));
  parts.reduce((offset, part) => (png.set(part, offset), offset + part.length), 0);
  return png;
}

In [ ]:
const width = 320, height = 160;
const pixels = new Uint8Array(width * height * 3);
for (let y = 0; y < height; y++) {
  for (let x = 0; x < width; x++) {
    const v = Math.sin(x / 16) + Math.sin(y / 8) + Math.sin((x + y) / 24) + Math.sin(Math.hypot(x - 160, y - 80) / 12);
    const i = (y * width + x) * 3;
    pixels[i] = 128 + 127 * Math.sin(v * Math.PI);
    pixels[i + 1] = 128 + 127 * Math.sin(v * Math.PI + 2);
    pixels[i + 2] = 128 + 127 * Math.sin(v * Math.PI + 4);
  }
}
display.png(encodePng(width, height, pixels));

## Updating outputs

With an `id`, `display` returns a handle to update the output later. `clearOutput` removes the outputs of the
cell, here to animate a countdown.

In [ ]:
const progressBar = percent => `<div style="width: 300px; background: #eee; border-radius: 4px">
  <div style="width: ${percent * 3}px; background: #4e79a7; color: white; padding: 2px 6px; border-radius: 4px">${percent}%</div>
</div>`;

const progress = display.html(progressBar(0), { id: true });
for (let percent = 10; percent <= 100; percent += 10) {
  await new Promise(resolve => setTimeout(resolve, 150));
  progress.update(progressBar(percent));
}

In [ ]:
for (let i = 3; i > 0; i--) {
  clearOutput(true);
  display.markdown(`# ${i}`);
  await new Promise(resolve => setTimeout(resolve, 500));
}
clearOutput(true);
display.markdown('# 🚀');